In [1]:
from collections import Counter, defaultdict
from gliner import GLiNER
import json
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import WordNetLemmatizer
import numpy as np
from pathlib import Path
from pyopenalex import OpenAlex
import PyPDF2
import re
import requests
import yake

In [2]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Александр\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Александр\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Александр\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Александр\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [3]:
# - "urchade/gliner_medium-v2.1" — универсальная, хорошо работает с английским
# - "nesemenpolkov/ruGliner-bookMeta" — специализированная на русские книги/статьи
# - "knowledgator/gliner-multitask-large-v0.5" — мультиязычная
model_name = "urchade/gliner_medium-v2.1"
model_path = Path("../models") / model_name
if not model_path.exists():
    model = GLiNER.from_pretrained("urchade/gliner_medium-v2.1")
    model.save_pretrained(model_path)
else:
    model = GLiNER.from_pretrained(model_path)

In [4]:
client = OpenAlex()

# Учебное заведение:

In [5]:
def institution_by_name(name: str):
    institutions = client.institutions.search(name).get()
    inst = institutions.results[0]

    print(f"ID: {inst.id}")
    print(f"Название: {inst.display_name}")
    print(f"Страна: {inst.country_code}")
    print(f"Рейтинг: {inst.works_count} работ, {inst.cited_by_count} цитирований")
    print(f"Сайт: {inst.homepage_url}")
    print(f"ROR ID: {inst.ror}")

    return inst

In [6]:
institutions = client.institutions.search("University of Tyumen").get()

In [7]:
inst = institution_by_name("University of Tyumen")

ID: https://openalex.org/I3020440027
Название: University of Tyumen
Страна: RU
Рейтинг: 12663 работ, 92242 цитирований
Сайт: https://www.utmn.ru/
ROR ID: https://ror.org/05vehv290


In [8]:
institution_by_name("Tyumen State University");

ID: https://openalex.org/I4210165799
Название: Tyumen State Medical University
Страна: RU
Рейтинг: 2866 работ, 20779 цитирований
Сайт: http://www.tyumsmu.ru/eng/
ROR ID: https://ror.org/05qbwsp96


In [9]:
institution_by_name("ТюмГУ");

ID: https://openalex.org/I3020440027
Название: University of Tyumen
Страна: RU
Рейтинг: 12663 работ, 92242 цитирований
Сайт: https://www.utmn.ru/
ROR ID: https://ror.org/05vehv290


# Работы:

In [10]:
def get_works(limit=5, years="2023|2024|2025"):
    """ Get list of works.
    """
    return client.works                                       \
        .filter(**{"authorships.institutions.ror": inst.ror}) \
        .filter(publication_year=years)                       \
        .limit(limit)                                         \
        .get(all=True).results

In [11]:
def unpack_abstract(abstract_inverted_index):
    """ Преобразует inverted index из OpenAlex в обычный текст.
    """
    if not abstract_inverted_index:
        return None

    # Находим максимальную позицию
    max_position = 0
    for positions in abstract_inverted_index.values():
        for pos in positions:
            if pos > max_position:
                max_position = pos

    # Создаем массив для текста
    text_array = [""] * (max_position + 1)

    # Заполняем массив словами
    for word, positions in abstract_inverted_index.items():
        for pos in positions:
            text_array[pos] = word

    # Собираем текст
    text = " ".join(text_array)

    # Исправляем пунктуацию
    text = re.sub(r' ([.,;:!?])', r'\1', text)
    text = re.sub(r' ([\)\]}])', r'\1', text)
    text = re.sub(r'([\(\[\{]) ', r'\1', text)
    text = re.sub(r'\s+', ' ', text)  # Убираем лишние пробелы

    return text.strip()

In [12]:
def get_best_title(work, text=None):
    """ Get best title.
    """
    # check for cyrillic in title
    title = work.title if hasattr(work, 'title') else ''
    if re.search(r'[а-яА-Я]', title):
        return title, 'ru'

    # check for cyrillic in display_name
    if hasattr(work, 'display_name') and work.display_name:
        if re.search(r'[а-яА-Я]', work.display_name):
            return work.display_name, 'ru'

    # check for title translations
    if hasattr(work, 'title_translations'):
        for lang, translation in work.title_translations.items():
            if lang == 'ru':
                return translation, 'ru'

    # if there is PDF text, search in it
    if text:
        if title := extract_title(text):
            return title, detect_language(title)

    return work.title, detect_language(work.title)

In [45]:
def extract_title(text):
    """ Try extracting work title from its text.
    """
    if not text:
        return None

    print('='*20)

    labels = ["название научной статьи"]

    # Разбиваем текст на предложения для более точного поиска
    # Берём первые 300 строк (где обычно находится название)
    lines = text.split('\n')[:300]

    for line in lines:
        line = line.strip()
        if not line or len(line) < 10:
            continue
        
        # Проверяем, что строка выглядит как название
        # (содержит русские буквы, начинается с заглавной)
        if not re.search(r'[а-яА-Я]', line):
            continue
        
        # if not re.match(r'^[А-Я]', line):
            # continue

        try:
            entities = model.predict_entities(line, labels, threshold=0.3)
            # if entities:
                # print(line)
                # print(entities)
            for entity in entities:
                if entity['label'] in labels:
                    title = entity['text'].strip()
                    title = re.sub(r'\s+', ' ', title)
                    if len(title) > 10 and len(title) < 300:
                        if detect_language(title) == 'ru':
                            return title
        except Exception as e:
            continue
    return None

In [14]:
def detect_language(text):
    """ Detect language of the text.
    """
    russian_chars = len(re.findall(r'[а-яА-Я]', text))
    english_chars = len(re.findall(r'[a-zA-Z]', text))

    if russian_chars == 0 and english_chars == 0:
        return 'unknown'

    if russian_chars > english_chars * 1.5:
        return 'ru'
    elif english_chars > russian_chars * 1.5:
        return 'en'
    else:
        # Смешанный случай — проверяем наличие русских слов
        if russian_chars > 10:
            return 'ru'
        return 'en'

In [15]:
def describe_work(work, index=None):
    """ Describe a piece of work.
    """
    title, lang = get_best_title(work)
    print(f"Статья {index if index is not None else ''}")
    print(f" - Название: {title}")
    print(f" - Год: {work.publication_year}")
    print(f" - Цитирований: {work.cited_by_count}")
    print(f" - ID: {work.id}")
    print(f" - DOI: {work.doi}")
    if work.open_access and work.open_access.is_oa:
            print(f" - Открытый доступ: {work.open_access.oa_url}")

    institutions_list = []
    authors = []
    utmn_authors = []
    for authorship in work.authorships:
        authors.append(authorship.author.display_name)
        for inst_in_work in authorship.institutions:
            institutions_list.append(inst_in_work.display_name)
            if inst_in_work.display_name == "University of Tyumen":
                utmn_authors.append(authorship.author.display_name)
    unique_institutions = list(set(institutions_list))

    print(f" - Институты: {len(unique_institutions)} всего")
    print(f" - Авторы: {len(authors)} всего")
    print(f" - Авторы из ТюмГУ: {', '.join(utmn_authors)}")

    domains = []
    fields = []
    subfields = []
    topics = []
    for topic in work.topics:
        domains.append(topic.domain.get('display_name'))
        fields.append(topic.field.get('display_name'))
        subfields.append(topic.subfield.get('display_name'))
        topics.append(topic.display_name)

    print(f" - Домен: {', '.join(list(set(domains)))}")
    print(f" - Поле: {', '.join(list(set(fields)))}")
    print(f" - Подполе: {', '.join(list(set(subfields)))}")
    print(f" - Тема: {', '.join(list(set(topics)))}")

    print(f" - Аннотация: \"{unpack_abstract(work.abstract_inverted_index)}\"")

    print()

In [16]:
def dump_works(works, titles, keywords, output_path):
    """ Dump info about works as JSON.
    """
    data = [
        {
            "title": titles[n],
            "title_en": work.title,
            "authors": [authorship.author.display_name for authorship in work.authorships],
            "utmn_authors": [
                authorship.author.display_name
                for authorship in work.authorships
                if any(map(lambda i: i.display_name == "University of Tyumen", authorship.institutions))
            ],
            "year": work.publication_year,
            "abstract": unpack_abstract(work.abstract_inverted_index),
            "doi": work.doi,
            "source_url": work.open_access.oa_url,
            "keywords": keywords[n] if n in keywords else None
        }
        for n, work in enumerate(works)
    ]
    output_file = Path(output_path)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    with open(output_file, 'w', encoding='utf-8') as file:
        json.dump(data, file, ensure_ascii=False, indent=2)

# Ключевые слова

In [17]:
def download_pdf(work, index, total, output_folder, timeout=30):
    """ Download work as PDF if available.
    """
    if not work.open_access or not work.open_access.is_oa:
        return None

    # define best link to url 
    pdf_url = None

    if hasattr(work, 'best_oa_location') and work.best_oa_location:
        if hasattr(work.best_oa_location, 'pdf_url') and work.best_oa_location.pdf_url:
            pdf_url = work.best_oa_location.pdf_url
        elif hasattr(work.best_oa_location, 'landing_page_url') and work.best_oa_location.landing_page_url:
            pdf_url = work.best_oa_location.landing_page_url
        elif hasattr(work.best_oa_location, 'url') and work.best_oa_location.url:
            pdf_url = work.best_oa_location.url

    if not pdf_url and work.open_access.oa_url:
        pdf_url = work.open_access.oa_url

    if not pdf_url and hasattr(work, 'locations'):
        for location in work.locations:
            if location.is_oa:
                if hasattr(location, 'pdf_url') and location.pdf_url:
                    pdf_url = location.pdf_url
                    break
                elif hasattr(location, 'landing_page_url') and location.landing_page_url:
                    if location.landing_page_url.endswith('.pdf') or 'pdf' in location.landing_page_url.lower():
                        pdf_url = location.landing_page_url
                        break

    if not pdf_url and hasattr(work, 'primary_location') and work.primary_location:
        if hasattr(work.primary_location, 'pdf_url') and work.primary_location.pdf_url:
            pdf_url = work.primary_location.pdf_url
        elif hasattr(work.primary_location, 'landing_page_url') and work.primary_location.landing_page_url:
            pdf_url = work.primary_location.landing_page_url
    
    if not pdf_url:
        return None

    if work.doi:
        doi_clean = work.doi.replace('https://doi.org/', '').replace('/', '_')
        filename = f"{doi_clean}.pdf"
    else:
        work_id = work.id.split('/')[-1]
        filename = f"{work_id}.pdf"

    filename = re.sub(r'[<>:"/\\|?*]', '_', filename)

    output_path = Path(output_folder) / filename
    output_path.parent.mkdir(parents=True, exist_ok=True)

    print(f"[{index}/{total}] Скачивание: {pdf_url}")

    if output_path.exists():
        print(f"(+) PDF уже существует: {filename}")
        return str(output_path)

    # download
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }

        response = requests.get(pdf_url, headers=headers, timeout=timeout, stream=True, allow_redirects=True)

        if response.status_code == 200:
            # check that this is actually PDF
            content_type = response.headers.get('content-type', '').lower()

            if 'pdf' not in content_type and not pdf_url.endswith('.pdf'):
                print(f"(-) Не PDF, а HTML: {filename}")
                return None

            with open(output_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)

            # check for minimum PDF size
            if output_path.stat().st_size < 1024:  # less than 1KB
                print(f"(-) Файл слишком маленький: {filename}")
                output_path.unlink()  # delete
                return None

            print(f"(+) PDF скачан: {filename} ({output_path.stat().st_size / 1024:.1f} KB)")
            return str(output_path)
        else:
            print(f"(-) Ошибка {response.status_code}: {filename}")
            return None

    except requests.exceptions.Timeout:
        print(f"(-) Таймаут: {filename}")
        return None
    except Exception as e:
        print(f"(-) Ошибка загрузки: {filename} - {e}")
        return None

In [18]:
def file_extract_text(path):
    """ Extract text fron a PDF file.
    """
    try:
        text = ""
        with open(path, 'rb') as file:
            reader = PyPDF2.PdfReader(file)
            for page in reader.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
        return text
    except Exception as e:
        print(f"Ошибка при чтении PDF {path}: {e}")
        return ""

In [19]:
def text_preprocess(text):
    """ Preprocess text: tokenize, lemmatize, remove stopwords.
    """
    if not text:
        return []

    sentences = sent_tokenize(text)
    words = []

    stop_words = set(stopwords.words('english'))

    # stopwords specific for articles
    stop_words.update([
        'et', 'al', 'fig', 'table', 'figure', 'equation', 'however',
        'therefore', 'thus', 'also', 'using', 'used', 'shown', 'show',
        'shows', 'fig', 'ref', 'reference', 'one', 'two', 'three',
        'may', 'can', 'will', 'would', 'could', 'should', 'might',
        'abstract', 'introduction', 'method', 'result', 'conclusion',
        'study', 'paper', 'research', 'analysis', 'data', 'results',
        'found', 'shown', 'observed', 'increased', 'decreased',
        'wa', 'doi', 'http', 'property', 'sci'
    ])

    lemmatizer = WordNetLemmatizer()

    for sent in sentences:
        tokens = word_tokenize(sent.lower())
        tokens = [token for token in tokens if token.isalpha() and len(token) > 2]
        tokens = [lemmatizer.lemmatize(token) for token in tokens]
        tokens = [token for token in tokens if token not in stop_words]
        words.extend(tokens)

    return words

In [20]:
def text_keywords_tfidf(words, top_n=20):
    """ Extract keywords from text using TF-IDF.
    """
    if not words:
        return []

    # count word frequency
    word_freq = Counter(words)

    # remove too frequent or rare words
    total_words = len(words)
    filtered_words = {
        word: count for word, count in word_freq.items()
        if 0.001 < count / total_words < 0.5  # between 0.1% and 50%
    }

    sorted_words = sorted(filtered_words.items(), key=lambda x: x[1], reverse=True)

    return [word for word, _ in sorted_words[:top_n]]

In [21]:
def text_keywords_rake(words, top_n=20):
    """ Extract keywords from text using RAKE.
    """
    if not words:
        return []

    # bigrams and trigrams
    bigrams = [' '.join(words[i:i+2]) for i in range(len(words)-1)]
    trigrams = [' '.join(words[i:i+3]) for i in range(len(words)-2)]
    
    # frequency of all phrases
    all_phrases = words + bigrams + trigrams
    phrase_freq = Counter(all_phrases)

    # filter out too long phrases
    filtered_phrases = {
        phrase: count for phrase, count in phrase_freq.items()
        if count > 1 and len(phrase.split()) <= 3
    }

    sorted_phrases = sorted(filtered_phrases.items(), key=lambda x: x[1], reverse=True)

    return [phrase for phrase, _ in sorted_phrases[:top_n]]

In [22]:
def text_keywords_textrank(words, top_n=20, window_size=2, iterations=30, damping=.85):
    """ Extract keywords from text using TextRank (PageRank для текста).
    """
    if not words:
        return []

    if len(words) < 3:
        return words[:top_n] if words else []

    # Убираем дубликаты для построения графа (сохраняем порядок)
    unique_words = []
    seen = set()
    for word in words:
        if word not in seen:
            unique_words.append(word)
            seen.add(word)

    if len(unique_words) < 3:
        return unique_words[:top_n]

    # Строим граф связей между словами
    word_to_index = {word: i for i, word in enumerate(unique_words)}
    graph = np.zeros((len(unique_words), len(unique_words)))

    # Проходим по тексту и строим связи между словами в окне
    for i in range(len(words)):
        for j in range(i + 1, min(i + window_size + 1, len(words))):
            if words[i] in word_to_index and words[j] in word_to_index:
                idx_i = word_to_index[words[i]]
                idx_j = word_to_index[words[j]]
                graph[idx_i][idx_j] += 1
                graph[idx_j][idx_i] += 1

    # Нормализуем граф
    for i in range(len(graph)):
        row_sum = graph[i].sum()
        if row_sum > 0:
            graph[i] = graph[i] / row_sum

    # Алгоритм PageRank
    scores = np.ones(len(unique_words)) / len(unique_words)

    for _ in range(iterations):
        new_scores = (1 - damping) / len(unique_words) + damping * scores @ graph
        # Проверяем сходимость
        if np.linalg.norm(new_scores - scores) < 1e-6:
            break
        scores = new_scores

    # Сортируем слова по важности
    word_scores = [(unique_words[i], scores[i]) for i in range(len(unique_words))]
    word_scores.sort(key=lambda x: x[1], reverse=True)

    # Возвращаем топ-N слов
    result = [word for word, _ in word_scores[:top_n]]

    return result

In [23]:
def text_keywords_yake(text, top_n=20, language="en", ngram_size=3, dedup_threshold=.9):
    """ Extract keywords from text using YAKE.
    """
    if not text:
        return []

    kw_extractor = yake.KeywordExtractor(
        lan=language,
        n=ngram_size,
        dedupLim=dedup_threshold,
        top=top_n
    )

    keywords = kw_extractor.extract_keywords(text)

    result = [kw for kw, score in keywords]

    return result

In [24]:
def text_keywords_combined(text, top_n=20):
    prep = text_preprocess(text)

    kw_tfidf = text_keywords_tfidf(prep, top_n*2)
    kw_rake = text_keywords_rake(prep, top_n*2)
    kw_textrank = text_keywords_textrank(prep, top_n*2)
    # kw_yake = text_keywords_yake(text, top_n*2)

    word_weights = defaultdict(float)

    for idx, word in enumerate(kw_tfidf):
        weight = 1.0 - (idx / len(kw_tfidf)) if kw_tfidf else 0
        word_weights[word] += weight

    for idx, word in enumerate(kw_rake):
        weight = 1.0 - (idx / len(kw_rake)) if kw_rake else 0
        word_weights[word] += weight

    for idx, word in enumerate(kw_textrank):
        weight = 1.0 - (idx / len(kw_textrank)) if kw_textrank else 0
        word_weights[word] += weight

    sorted_words = sorted(word_weights.items(), key=lambda x: x[1], reverse=True)

    return [word for word, _ in sorted_words[:top_n]]

# Таксономия:

In [25]:
def build_taxonomy_flat(works):
    """ Build flat taxonomy of topics.
    """
    domains = []
    fields = []
    subfields = []
    topics = []
    for work in works:
        for topic in work.topics:
            domains.append(topic.domain.get('display_name'))
            fields.append(topic.field.get('display_name'))
            subfields.append(topic.subfield.get('display_name'))
            topics.append(topic.display_name)
    return list(set(domains)), list(set(fields)), list(set(subfields)), list(set(topics))

In [26]:
def build_taxonomy_hierarchy(works):
    """ Build hierarchical taxonomy of topics.
    """
    hierarchy = {}

    for work in works:
        if not hasattr(work, 'topics') or not work.topics:
            continue

        for topic in work.topics:
            domain_name = topic.domain.get('display_name')
            field_name = topic.field.get('display_name')
            subfield_name = topic.subfield.get('display_name')
            topic_name = topic.display_name

            if domain_name not in hierarchy:
                hierarchy[domain_name] = {
                    'name': domain_name,
                    'level': 1,
                    'fields': {}
                }

            if field_name not in hierarchy[domain_name]['fields']:
                hierarchy[domain_name]['fields'][field_name] = {
                    'name': field_name,
                    'level': 2,
                    'subfields': {}
                }

            if subfield_name not in hierarchy[domain_name]['fields'][field_name]['subfields']:
                hierarchy[domain_name]['fields'][field_name]['subfields'][subfield_name] = {
                    'name': subfield_name,
                    'level': 3,
                    'topics': []
                }

            if topic_name not in hierarchy[domain_name]['fields'][field_name]['subfields'][subfield_name]['topics']:
                hierarchy[domain_name]['fields'][field_name]['subfields'][subfield_name]['topics'].append(topic_name)

    return hierarchy

In [27]:
def print_taxonomy_hierarchy(hierarchy):
    """ Print hierarchical taxonomy of topics.
    """
    for domain_idx, (domain_name, domain_data) in enumerate(hierarchy.items(), 1):
        print(f" ├ Домен {domain_idx} - '{domain_name}'")
        print(f" │ Содержит полей: {len(domain_data['fields'])}")
        print(f" │ ")

        for field_idx, (field_name, field_data) in enumerate(domain_data['fields'].items(), 1):
            print(f" ├──┬ Поле {field_idx} - '{field_name}'")
            print(f" │  │ Содержит подполей: {len(field_data['subfields'])}")
            print(f" │  │ ")

            for subfield_idx, (subfield_name, subfield_data) in enumerate(field_data['subfields'].items(), 1):
                print(f" │  ├──┬ Подполе {subfield_idx} - '{subfield_name}'")
                print(f" │  │  │ Содержит тем: {len(subfield_data['topics'])}")
                print(f" │  │  │ ")

                for topic_idx, topic_name in enumerate(subfield_data['topics'], 1):
                    print(f" │  │  ├─── Тема {topic_idx}: {topic_name}")
                    print(f" │  │  │ ")

        print("═" * 60)

In [28]:
def save_taxonomy_hierarchy(hierarchy, output_path):
    """ Save hierarchical taxonomy of topics as Markdown document.
    """
    output_file = Path(output_path)
    output_file.parent.mkdir(parents=True, exist_ok=True)

    lines = []
    lines.append("# Таксономия статей ТюмГУ")
    lines.append("")

    lines.append("## Содержание")
    lines.append("")
    for domain_idx, (domain_name, _) in enumerate(hierarchy.items(), 1):
        lines.append(f"{domain_idx}. [{domain_name}](#{domain_name.lower().replace(' ', '-')})")
    lines.append("")
    lines.append("---")
    lines.append("")

    for domain_idx, (domain_name, domain_data) in enumerate(hierarchy.items(), 1):
        lines.append(f"## {domain_idx}. {domain_name}")
        lines.append("")

        for field_idx, (field_name, field_data) in enumerate(domain_data['fields'].items(), 1):
            lines.append(f"{domain_idx}.{field_idx}. {field_name}")
            lines.append("")

            subfields_list = list(field_data['subfields'].keys())
            for subfield_idx, subfield_name in enumerate(subfields_list, 1):
                topics_count = len(field_data['subfields'][subfield_name]['topics'])
                lines.append(f"- {domain_idx}.{field_idx}.{subfield_idx}. {subfield_name}")
            lines.append("")

        lines.append("")

    lines.append("## Статистика")
    lines.append("")

    total_domains = len(hierarchy)
    total_fields = sum(len(domain['fields']) for domain in hierarchy.values())
    total_subfields = sum(
        sum(len(field['subfields']) for field in domain['fields'].values())
        for domain in hierarchy.values()
    )

    lines.append(f"- **Домены:** {total_domains}")
    lines.append(f"- **Области:** {total_fields}")
    lines.append(f"- **Подтемы:** {total_subfields}")
    lines.append("")
    lines.append("")
    lines.append("| Домен | Области | Подтемы |")
    lines.append("|-------|---------|---------|")

    for domain_name, domain_data in hierarchy.items():
        domain_fields = len(domain_data['fields'])
        domain_subfields = sum(len(field['subfields']) for field in domain_data['fields'].values())
        lines.append(f"| {domain_name} | {domain_fields} | {domain_subfields} |")

    with open(output_file, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))

In [29]:
def assign_taxonomy_tags(work, taxonomy):
    """ Assign tags based on taxonomy.
    """
    if not taxonomy:
        return []
    _, _, subfielfds, _ = build_taxonomy_flat([work])
    return subfielfds

# Главная функция:

In [30]:
def main():
    N = 100

    works = get_works(N)

    for n, work in enumerate(works):
        describe_work(work, n + 1)
    print()

    domains, fields, subfields, topics = build_taxonomy_flat(works)
    print(f"Домены ({len(domains)}): {', '.join(domains)}")
    print(f"Поля ({len(fields)}): {', '.join(fields)}")
    print(f"Подполя ({len(subfields)}): {', '.join(subfields)}")
    print(f"Темы ({len(topics)}): {', '.join(topics)}")
    print()

    hierarchy = build_taxonomy_hierarchy(works)
    print_taxonomy_hierarchy(hierarchy)
    save_taxonomy_hierarchy(hierarchy, "../docs/taxonomy.md")
    with open("../docs/taxonomy.json", 'w', encoding='utf-8') as file:
        json.dump(hierarchy, file, ensure_ascii=False, indent=2)

    downloaded_paths = {}
    pdf_texts = {}
    for n, work in enumerate(works):
        if path := download_pdf(work, n + 1, N, "../data/pdfs"):
            downloaded_paths[n] = path
            pdf_texts[n] = file_extract_text(path)
    print(f"Скачано статей: {len(downloaded_paths)}")

    keywords = {}
    for n in range(len(works)):
        kws = assign_taxonomy_tags(works[n], hierarchy)
        if n in downloaded_paths:
            text = file_extract_text(downloaded_paths[n])
            kws += text_keywords_combined(text)
        keywords[n] = list(set(kws)) if kws else None

    in_english = in_russian = 0
    true_titles = {}
    russian_titles = {}
    for n in range(N):
        title, lang = get_best_title(works[n], pdf_texts.get(n))
        true_titles[n] = title
        match lang:
            case 'ru':
                in_russian += 1
                russian_titles[n] = title
            case 'en':
                in_english += 1

    print('='*20)
    print("Работ на английском:", in_english)
    print("Работ на русском:", in_russian)

    for n, title in russian_titles.items():
        print(f"Работа {works[n].title}")
        print(f"Русское название: {title}")

    dump_works(works, true_titles, keywords, "../data/raw/openalex.json")

In [33]:
main()

Статья 1
 - Название: Addressing climate change with behavioral science: A global intervention tournament in 63 countries
 - Год: 2024
 - Цитирований: 255
 - ID: https://openalex.org/W4391629219
 - DOI: https://doi.org/10.1126/sciadv.adj5778
 - Открытый доступ: https://www.science.org/doi/pdf/10.1126/sciadv.adj5778?download=true
 - Институты: 101 всего
 - Авторы: 100 всего
 - Авторы из ТюмГУ: 
 - Домен: Social Sciences, Physical Sciences
 - Поле: Environmental Science, Psychology, Social Sciences
 - Подполе: Sociology and Political Science, Management, Monitoring, Policy and Law, Applied Psychology
 - Тема: Environmental Education and Sustainability, Behavioral Health and Interventions, Climate Change Communication and Perception
 - Аннотация: "Effectively reducing climate change requires marked, global behavior change. However, it is unclear which strategies are most likely to motivate people to change their climate beliefs and behaviors. Here, we tested 11 expert-crowdsourced interve

In [31]:
works = get_works(100)

In [32]:
domains, fields, subfields, topics = build_taxonomy_flat(works)

In [33]:
hierarchy = build_taxonomy_hierarchy(works)

In [34]:
downloaded_paths = {}
pdf_texts = {}
for n, work in enumerate(works):
    if path := download_pdf(work, n + 1, 100, "../data/pdfs"):
        downloaded_paths[n] = path
        pdf_texts[n] = file_extract_text(path)
print(f"Скачано статей: {len(downloaded_paths)}")

[1/100] Скачивание: https://www.science.org/doi/pdf/10.1126/sciadv.adj5778?download=true
(-) Ошибка 403: 10.1126_sciadv.adj5778.pdf
[2/100] Скачивание: https://resolver.sub.uni-goettingen.de/purl?gro-2/139675
(-) Ошибка 403: 10.1016_j.soilbio.2023.109223.pdf
[3/100] Скачивание: https://doi.org/10.1016/j.geoderma.2024.116831
(-) Не PDF, а HTML: 10.1016_j.geoderma.2024.116831.pdf
[5/100] Скачивание: https://doi.org/10.1016/j.geoderma.2023.116496
(-) Не PDF, а HTML: 10.1016_j.geoderma.2023.116496.pdf
[6/100] Скачивание: https://resolver.sub.uni-goettingen.de/purl?gro-2/143464
(-) Ошибка 403: 10.1021_acs.est.3c10247.pdf
[8/100] Скачивание: https://resolver.sub.uni-goettingen.de/purl?gro-2/139671
(-) Ошибка 403: 10.1016_j.scitotenv.2023.169423.pdf
[10/100] Скачивание: https://link.springer.com/content/pdf/10.1007/s44246-023-00051-7.pdf
(+) PDF уже существует: 10.1007_s44246-023-00051-7.pdf
[12/100] Скачивание: https://elar.urfu.ru/bitstream/10995/130681/1/2-s2.0-85146458208.pdf
(+) PDF уже 

In [36]:
keywords = {}
for n in range(len(works)):
    kws = assign_taxonomy_tags(works[n], hierarchy)
    if n in downloaded_paths:
        text = file_extract_text(downloaded_paths[n])
        kws += text_keywords_combined(text)
    keywords[n] = list(set(kws)) if kws else None

Ошибка при чтении PDF ..\data\pdfs\10.25259_sni_1014_2023.pdf: EOF marker not found


In [46]:
in_english = in_russian = 0
true_titles = {}
russian_titles = {}
for n in range(100):
    title, lang = get_best_title(works[n], pdf_texts.get(n))
    true_titles[n] = title
    match lang:
        case 'ru':
            in_russian += 1
            russian_titles[n] = title
        case 'en':
            in_english += 1

In [47]:
russian_titles

{12: 'Соматическая дисфункция', 84: 'Методология'}

In [46]:
true_titles

{0: 'Addressing climate change with behavioral science: A global intervention tournament in 63 countries',
 1: 'Nitrogen fertilizer builds soil organic carbon under straw return mainly via microbial necromass formation',
 2: 'Inorganic carbon is overlooked in global soil carbon research: A bibliometric analysis',
 3: 'Carbon stabilization pathways in soil aggregates during long-term forest succession: Implications from δ13C signatures',
 4: 'Organic carbon accumulation and microbial activities in arable soils after abandonment: A chronosequence study',
 5: 'Do Added Microplastics, Native Soil Properties, and Prevailing Climatic Conditions Have Consequences for Carbon and Nitrogen Contents in Soil? A Global Data Synthesis of Pot and Greenhouse Studies',
 6: 'Long-term organic fertilizer-induced carbonate neoformation increases carbon sequestration in soil',
 7: 'Soil organic matter turnover: Global implications from δ13C and δ15N signatures',
 8: 'Site Occupation Engineering toward Gian

In [56]:
for k, v in pdf_texts.items():
    print(k, v)
    print('='*20)

9 Deng  et al. Carbon Research            (2023) 2:19  
https://doi.org/10.1007/s44246-023-00051-7
ORIGINAL ARTICLE Open Access
© The Author(s) 2023. Open Access This article is licensed under a Creative Commons Attribution 4.0 International License, which 
permits use, sharing, adaptation, distribution and reproduction in any medium or format, as long as you give appropriate credit to the 
original author(s) and the source, provide a link to the Creative Commons licence, and indicate if changes were made. The images or 
other third party material in this article are included in the article’s Creative Commons licence, unless indicated otherwise in a credit line 
to the material. If material is not included in the article’s Creative Commons licence and your intended use is not permitted by statutory 
regulation or exceeds the permitted use, you will need to obtain permission directly from the copyright holder. To view a copy of this 
licence, visit http:// creat iveco mmons. org/ licen 

In [37]:
pdf_texts[12]

'8Российский остеопатический журнал\n2023; 2: 8–90Russian Osteopathic Journal\n2023; 2: 8–90\nУДК 615.828+616-008.6https://doi.org/10.32885/2220-0975-2023-2-8-90© Д. Е. Мохов, В. О. Белаш, И. А. Аптекарь, \nЭ. Н. Ненашкина, Ю. П. Потехина, \nЕ. С. Трегубова, А. Ф. Беляев, 2023 \nСоматическая дисфункция. \nКлинические рекомендации 2023\nКодирование по Международной статистической классификации болезней и проблем, \nсвязанных со здоровьем: М99.0, М99.8, М99.9Возрастная группа: взрослые, детиГод утверждения: 2023Разработчик клинических рекомендаций: Общероссийская общественная организация содействия развитию остеопатии «Российская остеопатическая ассоциация»Одобрены научно-практическим советом Министерства здравоохранения Российской Федерации (протокол от 27.12.2022 № 23) Утверждены правлением Общероссийской общественной организации содействия развитию остеопатии «Российская остеопатическая ассоциация» (протокол от 18.01.2023 №1/23)\nД. Е. Мохов 1, 2,*, В. О. Белаш 1, 6, И. А. Аптекарь 3,

In [44]:
get_best_title(works[12], pdf_texts.get(12))

связанных со здоровьем: М99.0, М99.8, М99.9Возрастная группа: взрослые, детиГод утверждения: 2023Разработчик клинических рекомендаций: Общероссийская общественная организация содействия развитию остеопатии «Российская остеопатическая ассоциация»Одобрены научно-практическим советом Министерства здравоохранения Российской Федерации (протокол от 27.12.2022 № 23) Утверждены правлением Общероссийской общественной организации содействия развитию остеопатии «Российская остеопатическая ассоциация» (протокол от 18.01.2023 №1/23)
[{'start': 24, 'end': 29, 'text': 'М99.0', 'label': 'название научной статьи', 'score': 0.3531922399997711}]
Беляев А. Ф. Соматическая дисфункция. Клинические рекомендации 2023. Российский остеопатический журнал. 2023; 2: 8–90. https://doi.org/10.32885/2220-0975-2023-2-8-90
[{'start': 63, 'end': 67, 'text': '2023', 'label': 'название научной статьи', 'score': 0.7271210551261902}, {'start': 104, 'end': 108, 'text': '2023', 'label': 'название научной статьи', 'score': 0.7

('Соматическая дисфункция', 'ru')